#Importing libraries

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import cv2
import seaborn as sns
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical

from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


#Data preparation

In [3]:
from google.colab import files

uploaded = files.upload()

Saving archive (9).zip to archive (9) (2).zip


In [ ]:
!unzip "/content/archive.zip" -d "/content/dataset"

Archive:  /content/archive.zip
replace /content/dataset/Testing/glioma/Te-gl_1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
os.listdir('/content/dataset')

In [ ]:
train_path = "/content/dataset/Training"
test_path = "/content/dataset/Testing"


print(os.listdir(train_path))

In [ ]:
for cls in os.listdir(train_path):
    print(
        cls,
        len(os.listdir(os.path.join(train_path, cls)))
    )


#Data preprocessing & Visualization

In [ ]:

classes = ['glioma', 'meningioma', 'pituitary', 'notumor']
counts = [1400, 1400, 1400, 1400]

plt.figure(figsize=(6,4))
plt.bar(classes, counts)

plt.title("Class Distribution")
plt.xlabel("Classes")
plt.ylabel("Number of Images")

plt.show()

In [ ]:
classes = os.listdir(train_path)

plt.figure(figsize=(12,4))

for i, cls in enumerate(classes):

    img_name = os.listdir(os.path.join(train_path, cls))[0]

    img_path = os.path.join(train_path, cls, img_name)

    img = Image.open(img_path)

    plt.subplot(1, len(classes), i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
classes = [
    'glioma',
    'meningioma',
    'notumor',
    'pituitary'
]

plt.figure(figsize=(12,4))

for i, cls in enumerate(classes):

    img_name = os.listdir(os.path.join(train_path, cls))[0]

    img_path = os.path.join(train_path, cls, img_name)

    img = Image.open(img_path)

    plt.subplot(1, len(classes), i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis('off')

plt.tight_layout()
plt.show()

#Image resize

In [ ]:
IMG_SIZE = 224

X = []
y = []

for label, cls in enumerate(classes):

    class_path = os.path.join(train_path, cls)

    for img_name in os.listdir(class_path):

        img_path = os.path.join(class_path, img_name)

        img = Image.open(img_path).convert('RGB')

        img = img.resize((IMG_SIZE, IMG_SIZE))

        img = np.array(img)

        X.append(img)

        y.append(label)

X = np.array(X, dtype="float32") / 255.0
y = np.array(y)
print(X.shape)
print(y.shape)



In [ ]:
cls = os.listdir(train_path)[0]

img_name = os.listdir(os.path.join(train_path, cls))[0]

img_path = os.path.join(train_path, cls, img_name)

img = cv2.imread(img_path)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
print(img.shape)

print(img.dtype)

print(img.min())

print(img.max())

# Data splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

#Convert Labels to Categorical

In [ ]:

y_train = to_categorical(y_train)
y_val = to_categorical(y_val)
y_test = to_categorical(y_test)

#Building CNN model

In [ ]:
model = Sequential([

    Conv2D(32, (3,3),
           activation='relu',
           input_shape=(224,224,3)),

    MaxPooling2D((2,2)),

    Conv2D(64, (3,3),
           activation='relu'),

    MaxPooling2D((2,2)),

    Conv2D(128, (3,3),
           activation='relu'),

    MaxPooling2D((2,2)),

    Flatten(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(4, activation='softmax')

])




In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=32
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
model.save('brain_tumor_cnn.keras')